In [1]:
import pandas as pd

def parse_dates_flexible(series):
    """
    Parses dates with mixed formats.
    Tries DD-MM-YYYY HH:MM first, then falls back to YYYY-MM-DD HH:MM:SS.
    """
    # Try first format: DD-MM-YYYY HH:MM
    parsed = pd.to_datetime(series, format='%d-%m-%Y %H:%M', errors='coerce')
    
    # Identify rows that failed to parse
    mask = parsed.isna()
    
    # Try second format on failures: YYYY-MM-DD HH:MM:SS
    if mask.any():
        parsed[mask] = pd.to_datetime(series[mask], format='%d-%m-%Y %H:%M', errors='coerce')
    
    return parsed

# 1. Load the datasets
df_events = pd.read_csv('events.csv')
df_events1 = pd.read_csv('events1.csv')

# 2. Convert 'Timestamp' column to datetime objects using the flexible parser
df_events['Timestamp'] = parse_dates_flexible(df_events['Timestamp'])
df_events1['Timestamp'] = parse_dates_flexible(df_events1['Timestamp'])

# 3. Concatenate the two DataFrames
df_merged = pd.concat([df_events, df_events1])

# 4. Sort the combined data by Timestamp
df_merged = df_merged.sort_values(by='Timestamp')

# 5. Save the result to a new CSV file
output_filename = 'merged_events_sorted.csv'
df_merged.to_csv(output_filename, index=False)

print(f"Successfully merged and sorted data into {output_filename}")

Successfully merged and sorted data into merged_events_sorted.csv


# Session to CSV Convert

In [18]:
import pandas as pd
import re

def parse_log_file(filepath):
    """
    Parses a log file and returns a list of dictionaries with Timestamp, Level, and Message.
    """
    data = []
    # Regex to match the log format: YYYY-MM-DD HH:MM:SS,mmm - LEVEL - Message
    # Example: 2025-11-28 13:03:11,913 - WARNING - Loaded 1 guards
    # FIXED: Changed group names to Title Case (Timestamp, Level, Message) to match DataFrame column usage
    log_pattern = re.compile(r'(?P<Timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2},\d{3}) - (?P<Level>\w+) - (?P<Message>.*)')
    
    with open(filepath, 'r') as f:
        for line in f:
            match = log_pattern.match(line.strip())
            if match:
                data.append(match.groupdict())
    return data

# 1. List of log files to process
log_files = ['session.log','session1.log','session2.log','session3.log','session4.log','session5.log']

# 2. Collect data from all files
all_log_data = []
for file in log_files:
    try:
        print(f"Processing {file}...")
        file_data = parse_log_file(file)
        all_log_data.extend(file_data)
    except FileNotFoundError:
        print(f"Warning: {file} not found. Skipping.")

# 3. Create DataFrame
if not all_log_data:
    print("Warning: No data found in log files. Creating empty DataFrame.")
    df_logs = pd.DataFrame(columns=['Timestamp', 'Level', 'Message'])
else:
    df_logs = pd.DataFrame(all_log_data)

# 4. Convert 'Timestamp' to datetime objects for accurate sorting
# Log format is YYYY-MM-DD HH:MM:SS,mmm (comma for milliseconds)
if not df_logs.empty:
    df_logs['Timestamp'] = pd.to_datetime(df_logs['Timestamp'], format='%Y-%m-%d %H:%M:%S,%f')

    # 5. Sort by Timestamp
    df_logs = df_logs.sort_values(by='Timestamp')

    # 6. STANDARDIZE the date format for the output CSV
    # Converting to "DD-MM-YYYY HH:MM:SS" (dropping milliseconds for cleaner look, or keep %f if needed)
    df_logs['Timestamp'] = df_logs['Timestamp'].dt.strftime('%d-%m-%Y %H:%M:%S')

# 7. Save to CSV
output_filename = 'merged_session_logs_file1.csv'
df_logs.to_csv(output_filename, index=False)

print(f"Successfully converted logs to CSV. Saved as {output_filename}")
print(df_logs.head())

Processing session.log...
Processing session1.log...
Processing session2.log...
Processing session3.log...
Processing session4.log...
Processing session5.log...
Successfully converted logs to CSV. Saved as merged_session_logs_file1.csv
             Timestamp    Level  \
0  28-11-2025 13:03:11  WARNING   
1  28-11-2025 13:03:48  WARNING   
2  28-11-2025 13:03:56  WARNING   
3  28-11-2025 13:03:57    ERROR   
4  28-11-2025 13:04:08  WARNING   

                                             Message  
0                                    Loaded 1 guards  
1                      Camera 2 started successfully  
2       Failed to read frame, attempting recovery...  
3  Failed to recover after 10 retries, attempting...  
4       Switched to Camera 0 - Restarting video feed  


# Basic info 
 

In [7]:
import pandas as pd

# Read the file
df = pd.read_csv('events.csv')

# Inspect the data
print(df.head())
print(df.info())

          Timestamp   Name               Action           Status  \
0  28-11-2025 14:27  Guard  'Hands Up' required          TIMEOUT   
1  28-11-2025 14:27  Guard             Standing  ALERT TRIGGERED   
2  28-11-2025 14:27  Guard             Standing  ALERT CONTINUED   
3  28-11-2025 14:27  Guard             Standing  ALERT CONTINUED   
4  28-11-2025 14:27  Guard  'Hands Up' required        PERFORMED   

                                        Image_Path  Confidence  
0  alert_snapshots\alert_Guard_20251128_142730.jpg        0.82  
1                                              NaN        0.53  
2                                              NaN        0.53  
3                                              NaN        0.53  
4                                              NaN        0.82  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 319 entries, 0 to 318
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Timestamp 

In [8]:
print("Unique Actions:")
print(df['Action'].unique())
print("\nUnique Statuses:")
print(df['Status'].unique())

Unique Actions:
["'Hands Up' required" 'Standing' 'FUGITIVE_DETECTED' 'STILLNESS_ALERT'
 'MISSING' 'Hands Crossed']

Unique Statuses:
['TIMEOUT' 'ALERT TRIGGERED' 'ALERT CONTINUED' 'PERFORMED'
 'FIRST_APPEARANCE (confidence: 0.475)' 'NO MOTION FOR 15.1 SECONDS'
 'FIRST_APPEARANCE (confidence: 0.354)' 'ALERT TRIGGERED - TARGET MISSING'
 'FIRST_APPEARANCE (confidence: 0.497)'
 'FIRST_APPEARANCE (confidence: 0.517)'
 'FIRST_APPEARANCE (confidence: 0.508)'
 'FIRST_APPEARANCE (confidence: 0.281)'
 'FIRST_APPEARANCE (confidence: 0.299)'
 'FIRST_APPEARANCE (confidence: 0.463)'
 'FIRST_APPEARANCE (confidence: 0.448)'
 'FIRST_APPEARANCE (confidence: 0.358)'
 'FIRST_APPEARANCE (confidence: 0.575)'
 'FIRST_APPEARANCE (confidence: 0.610)'
 'FIRST_APPEARANCE (confidence: 0.537)' 'NO MOTION FOR 15.2 SECONDS']


# . Download the Cleaned Data


In [19]:
import pandas as pd

# Load the first few rows of the first CSV file
df1 = pd.read_csv('merged_session_logs_file.csv')
print("First file info:")
print(df1.info())
print(df1.head())

# # Load the first few rows of the second CSV file
# df2 = pd.read_csv('merged_session_logs2.csv')
# print("\nSecond file info:")
# print(df2.info())
# print(df2.head())

First file info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7040 entries, 0 to 7039
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Timestamp  7040 non-null   object
 1   Level      7040 non-null   object
 2   Message    7040 non-null   object
dtypes: object(3)
memory usage: 165.1+ KB
None
             Timestamp    Level  \
0  28-11-2025 13:03:11  WARNING   
1  28-11-2025 13:03:48  WARNING   
2  28-11-2025 13:03:56  WARNING   
3  28-11-2025 13:03:57    ERROR   
4  28-11-2025 13:04:08  WARNING   

                                             Message  
0                                    Loaded 1 guards  
1                      Camera 2 started successfully  
2       Failed to read frame, attempting recovery...  
3  Failed to recover after 10 retries, attempting...  
4       Switched to Camera 0 - Restarting video feed  


In [20]:
import re

# patterns
# 1. Guard Name and Confidence from DETECTED messages
# Sample: [DETECTED] OK Rudresh identified & tracking (confidence: 0.700, distance: 0.300, bbox: 186x186 px)
# 2. Camera ID
# Sample: Camera 2 started successfully

def extract_info(row):
    message = row['Message']
    guard_name = None
    confidence = None
    camera_id = None
    event_type = "System"
    
    # Check for detection
    det_match = re.search(r"\[DETECTED\] OK (.*?) identified.*?confidence: (\d+\.\d+)", message)
    if det_match:
        guard_name = det_match.group(1)
        confidence = float(det_match.group(2))
        event_type = "Detection"
    
    # Check for Alert
    alert_match = re.search(r"\[ALERT\] (.*?):", message)
    if alert_match:
        guard_name = alert_match.group(1).replace(" bhai", "") # Cleaning honorifics if needed
        event_type = "Alert"

    # Check for Camera
    cam_match = re.search(r"Camera (\d+)", message)
    if cam_match:
        camera_id = cam_match.group(1)
        event_type = "CameraStatus"
        
    # Check for Error
    if "Failed" in message or "ERROR" in row['Level']:
        event_type = "Error"
        
    return pd.Series([guard_name, confidence, camera_id, event_type])

# concat for testing
df_all = pd.concat([df1])
# df_all = pd.concat([df1, df2])
df_all[['GuardName', 'Confidence', 'CameraID', 'EventType']] = df_all.apply(extract_info, axis=1)

print(df_all['EventType'].value_counts())
print(df_all[['Message', 'GuardName', 'Confidence']].dropna().head())
print(df_all[df_all['EventType'] == 'Alert'].head())


EventType
System          5586
Detection        789
Alert            363
Error            196
CameraStatus     106
Name: count, dtype: int64
                                              Message GuardName  Confidence
35  [DETECTED] OK Guard identified & tracking (con...     Guard       0.528
41  [DETECTED] OK Guard identified & tracking (con...     Guard       0.687
59  [DETECTED] OK Guard identified & tracking (con...     Guard       0.487
61  [DETECTED] OK Guard identified & tracking (con...     Guard       0.614
66  [DETECTED] OK Guard identified & tracking (con...     Guard       0.514
               Timestamp    Level  \
38   28-11-2025 14:27:30  WARNING   
39   28-11-2025 14:27:31  WARNING   
42   28-11-2025 14:29:36  WARNING   
43   28-11-2025 14:29:36  WARNING   
796  09-12-2025 10:51:59  WARNING   

                                              Message GuardName  Confidence  \
38   [ALERT] Guard: Action TIMEOUT - triggering alarm     Guard         NaN   
39           [ALERT] G

In [21]:
def refined_extract(row):
    message = row['Message']
    guard_name = None
    confidence = None
    event_category = "System Log" # Default
    details = None
    
    # 1. Detection
    # [DETECTED] OK Rudresh identified & tracking (confidence: 0.700...
    det_match = re.search(r"\[DETECTED\] OK (.*?) identified.*?confidence: (\d+\.\d+)", message)
    if det_match:
        guard_name = det_match.group(1).strip()
        confidence = float(det_match.group(2))
        event_category = "Detection"
        
    # 2. Tracking Start
    # [TRACKING START] Identifying and tracking: Rudresh | Action: Hands Up
    track_match = re.search(r"\[TRACKING START\] Identifying and tracking: (.*?) \| Action: (.*)", message)
    if track_match:
        guard_name = track_match.group(1).strip()
        details = track_match.group(2).strip()
        event_category = "Tracking Start"

    # 3. Alert
    # [ALERT] Rudresh bhai: Restarting 15s alarm loop
    # [ALERT] Rudresh: Action TIMEOUT - triggering alarm
    alert_match = re.search(r"\[ALERT\] (.*?): (.*)", message)
    if alert_match:
        guard_name = alert_match.group(1).replace(" bhai", "").strip()
        details = alert_match.group(2).strip()
        event_category = "Alert"
        
    # 4. Ghost Removed
    # [GHOST REMOVED] Rudresh: Body detected...
    if "[GHOST REMOVED]" in message:
        event_category = "Ghost Removed"
        # Try to get name
        ghost_match = re.search(r"\[GHOST REMOVED\] (.*?):", message)
        if ghost_match:
            guard_name = ghost_match.group(1).strip()
            
    # 5. Errors
    if row['Level'] == 'ERROR' or "Failed" in message:
        event_category = "Error"
        details = message
        
    # 6. Camera Status
    if "Camera" in message and "started" in message:
        event_category = "Camera Status"
        details = message
        
    return pd.Series([guard_name, confidence, event_category, details])

# Reload fresh to be safe
df1 = pd.read_csv('merged_session_logs_file.csv')
df_final = pd.concat([df1], ignore_index=True)

# # Reload fresh to be safe
# df1 = pd.read_csv('merged_session_logs.csv')
# df2 = pd.read_csv('merged_session_logs1.csv')
# df_final = pd.concat([df1, df2], ignore_index=True)

# Parse Timestamp
df_final['Timestamp'] = pd.to_datetime(df_final['Timestamp'], dayfirst=True)

# Apply extraction
df_final[['Guard_Name', 'Confidence_Score', 'Event_Category', 'Event_Details']] = df_final.apply(refined_extract, axis=1)

# Fill NaNs in text columns with "N/A" or empty string for cleaner PowerBI import? 
# Better to leave as NaN or specific text. Let's start with None -> "Unknown" for categories if needed, but PowerBI handles nulls.
# Let's save it.
df_final.to_csv('cleaned_session_logs_for_powerbi.csv', index=False)
print("File saved.")
print(df_final.head())
print(df_final['Event_Category'].value_counts())

File saved.
            Timestamp    Level  \
0 2025-11-28 13:03:11  WARNING   
1 2025-11-28 13:03:48  WARNING   
2 2025-11-28 13:03:56  WARNING   
3 2025-11-28 13:03:57    ERROR   
4 2025-11-28 13:04:08  WARNING   

                                             Message Guard_Name  \
0                                    Loaded 1 guards       None   
1                      Camera 2 started successfully       None   
2       Failed to read frame, attempting recovery...       None   
3  Failed to recover after 10 retries, attempting...       None   
4       Switched to Camera 0 - Restarting video feed       None   

   Confidence_Score Event_Category  \
0               NaN     System Log   
1               NaN  Camera Status   
2               NaN          Error   
3               NaN          Error   
4               NaN     System Log   

                                       Event_Details  
0                                               None  
1                      Camera 2 started s

# time-based analysis 

In [5]:
# Reload the data (since state might be lost or to be safe)
import pandas as pd
import re

# Re-defining the extraction function as per previous successful step
def refined_extract(row):
    message = row['Message']
    guard_name = None
    confidence = None
    event_category = "System Log" 
    details = None
    
    det_match = re.search(r"\[DETECTED\] OK (.*?) identified.*?confidence: (\d+\.\d+)", message)
    if det_match:
        guard_name = det_match.group(1).strip()
        confidence = float(det_match.group(2))
        event_category = "Detection"
        
    track_match = re.search(r"\[TRACKING START\] Identifying and tracking: (.*?) \| Action: (.*)", message)
    if track_match:
        guard_name = track_match.group(1).strip()
        details = track_match.group(2).strip()
        event_category = "Tracking Start"

    alert_match = re.search(r"\[ALERT\] (.*?): (.*)", message)
    if alert_match:
        guard_name = alert_match.group(1).replace(" bhai", "").strip()
        details = alert_match.group(2).strip()
        event_category = "Alert"
        
    if "[GHOST REMOVED]" in message:
        event_category = "Ghost Removed"
        ghost_match = re.search(r"\[GHOST REMOVED\] (.*?):", message)
        if ghost_match:
            guard_name = ghost_match.group(1).strip()
            
    if row['Level'] == 'ERROR' or "Failed" in message:
        event_category = "Error"
        details = message
        
    if "Camera" in message and "started" in message:
        event_category = "Camera Status"
        details = message
        
    return pd.Series([guard_name, confidence, event_category, details])

# Load original files
df1 = pd.read_csv('merged_session_logs.csv')
df2 = pd.read_csv('merged_session_logs1.csv')
df_final = pd.concat([df1, df2], ignore_index=True)

# Parse Timestamp
df_final['Timestamp'] = pd.to_datetime(df_final['Timestamp'], dayfirst=True)

# Apply extraction
df_final[['Guard_Name', 'Confidence_Score', 'Event_Category', 'Event_Details']] = df_final.apply(refined_extract, axis=1)

# NEW STEP: Extract Time Components
df_final['Hour'] = df_final['Timestamp'].dt.hour
df_final['Minute'] = df_final['Timestamp'].dt.minute
df_final['Second'] = df_final['Timestamp'].dt.second
df_final['Time_Str'] = df_final['Timestamp'].dt.strftime('%H:%M:%S') # Useful for display

# Save updated file
df_final.to_csv('cleaned_session_logs_with_time.csv', index=False)

print("New columns added: Hour, Minute, Second, Time_Str")
print(df_final[['Timestamp', 'Hour', 'Minute', 'Second']].head())
print(df_final.info())

New columns added: Hour, Minute, Second, Time_Str
            Timestamp  Hour  Minute  Second
0 2025-11-28 13:03:11    13       3      11
1 2025-11-28 13:03:48    13       3      48
2 2025-11-28 13:03:56    13       3      56
3 2025-11-28 13:03:57    13       3      57
4 2025-11-28 13:04:08    13       4       8
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3986 entries, 0 to 3985
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Timestamp         3986 non-null   datetime64[ns]
 1   Level             3986 non-null   object        
 2   Message           3986 non-null   object        
 3   Guard_Name        799 non-null    object        
 4   Confidence_Score  432 non-null    float64       
 5   Event_Category    3986 non-null   object        
 6   Event_Details     368 non-null    object        
 7   Hour              3986 non-null   int32         
 8   Minute            3986 non-null   